In [1]:
import qutip as qt
from qutip import tensor
import numpy as np
import matplotlib.pyplot as plt
from quantum_logical.channel import AmplitudeDamping, PhaseDamping
from quantum_logical.trotter import TrotterGroup
from tqdm import tqdm
from quantum_logical.operators import selective_destroy
from scipy.optimize import curve_fit

In [2]:
# generating parameters and creating initial state
# T1 = 50
# T2 = 25
T1 = 1
T2 = .5
N = 5
dim = 2
trotter_dt = .0001

amp_damp_channel = AmplitudeDamping(T1, num_qubits=N, hilbert_space_dim=dim)
phase_damp_channel = PhaseDamping(T1, T2, num_qubits=N, hilbert_space_dim=dim)
# trotter = TrotterGroup(
#     continuous_operators=[amp_damp_channel, phase_damp_channel],
#     trotter_dt=trotter_dt,
# )
trotter = TrotterGroup(
    continuous_operators=[amp_damp_channel],
    trotter_dt=trotter_dt,
)


In [17]:
# creating the set of cnots
# cnot1 = qt.cnot(N=5, control=0, target=1)
# cnot2 = qt.cnot(N=5, control=0, target=2)

# cnot3 = qt.cnot(N=5, control=0, target=3)
# cnot4 = qt.cnot(N=5, control=1, target=3)

# cnot5 = qt.cnot(N=5, control=1, target=4)
# cnot6 = qt.cnot(N=5, control=2, target=4)

x_gate = qt.rx(np.pi / 2, N=None)

z_gate_positive = qt.rz(np.pi / 2, N=None)
z_gate_negative = qt.rz(- np.pi / 2, N=None)

gate_set = [z_gate_negative, x_gate, z_gate_positive]
# this needs to be more automated 

def create_iswap(N, target, control):
    return qt.iswap(N=N, targets=[control, target])

# need to find a way for the computer to recognize the layers 

layer_one_list = []
layer_two_list = []
layer_three_list = []
layer_one_list.append(tensor( z_gate_negative, tensor([qt.qeye(dim)] * 2), z_gate_positive * x_gate, tensor([qt.qeye(dim)] * 1)))
layer_one_list.append(tensor(tensor([qt.qeye(dim)] * 1), z_gate_negative, tensor([qt.qeye(dim)] * 1), z_gate_positive * x_gate, tensor([qt.qeye(dim)] * 1)))
layer_one_list.append(tensor(tensor([qt.qeye(dim)] * 1), z_gate_negative, tensor([qt.qeye(dim)] * 2), z_gate_positive * x_gate))
layer_one_list.append(tensor(tensor([qt.qeye(dim)] * 2), z_gate_negative, tensor([qt.qeye(dim)] * 1), z_gate_positive * x_gate))

layer_two_list.append(tensor( x_gate, tensor([qt.qeye(dim)] * 2),qt.qeye(dim), tensor([qt.qeye(dim)] * 1)))
layer_two_list.append(tensor(tensor([qt.qeye(dim)] * 1), x_gate, tensor([qt.qeye(dim)] * 2),qt.qeye(dim), tensor([qt.qeye(dim)] * 1)))
layer_two_list.append(tensor(tensor([qt.qeye(dim)] * 1), x_gate, tensor([qt.qeye(dim)] * 2),qt.qeye(dim)))
layer_two_list.append(tensor(tensor([qt.qeye(dim)] * 2), x_gate, tensor([qt.qeye(dim)] * 2),qt.qeye(dim)))

layer_three_list.append(tensor( qt.qeye(dim), tensor([qt.qeye(dim)] * 2), z_gate_positive, tensor([qt.qeye(dim)] * 1)))
layer_three_list.append(tensor(tensor([qt.qeye(dim)] * 1), qt.qeye(dim), tensor([qt.qeye(dim)] * 2), z_gate_positive, tensor([qt.qeye(dim)] * 1)))
layer_three_list.append(tensor(tensor([qt.qeye(dim)] * 1), qt.qeye(dim), tensor([qt.qeye(dim)] * 2), z_gate_positive))
layer_three_list.append(tensor(tensor([qt.qeye(dim)] * 2), qt.qeye(dim), tensor([qt.qeye(dim)] * 2), z_gate_positive))



iswaps = []
i = 0
layers = []
for j in range(3,5):
    for i in range(i, i + 2):
        # print({i}, {j})
        iswaps.extend([create_iswap(N=N, control=i, target=j)] * 2)

gates_order = []
for i in range(len(layer_one_list)):
    gates_order.append(layer_one_list[i])
    gates_order.append(iswaps[(2 * i)])
    gates_order.append(layer_two_list[i])
    gates_order.append(iswaps[(2 * i)])
    gates_order.append(layer_three_list[i])


{0} {3}
{1} {3}
{1} {4}
{2} {4}


C:\Users\girgi\AppData\Local\Temp\ipykernel_17844\1151883667.py:11: DeprecationWarning: Importing functions/classes of the qip submodule directly from the namespace qutip is deprecated. Please import them from the submodule instead, e.g.
from qutip.qip.operations import cnot
from qutip.qip.circuit import QubitCircuit

  x_gate = qt.rx(np.pi / 2, N=None)
C:\Users\girgi\AppData\Local\Temp\ipykernel_17844\1151883667.py:13: DeprecationWarning: Importing functions/classes of the qip submodule directly from the namespace qutip is deprecated. Please import them from the submodule instead, e.g.
from qutip.qip.operations import cnot
from qutip.qip.circuit import QubitCircuit

  z_gate_positive = qt.rz(np.pi / 2, N=None)
C:\Users\girgi\AppData\Local\Temp\ipykernel_17844\1151883667.py:14: DeprecationWarning: Importing functions/classes of the qip submodule directly from the namespace qutip is deprecated. Please import them from the submodule instead, e.g.
from qutip.qip.operations import cnot
fro

In [ ]:
# built the list of gates not try to make the cnots and see if it works 
# this is the point at which you start the trotterization of the system 